In [ ]:
# Medical Insurance Charges Prediction - Linear Regression Analysis
# MACHINE LEARNING TECHNIQUES ASSIGNMENT 1 2024/A/KSD/1255/F
# Course: MACHINE LEARNING TECHNIQUES

# --- IMPORT LIBRARIES ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set a clean and modern style for better visualizations
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# --- PART A: DATA ACQUISITION AND UNDERSTANDING ---

# Load the dataset from a CSV file into a pandas DataFrame
df = pd.read_csv('/content/Medical Cost Personal Dataset.csv')

# Display the first 5 rows of the dataset to get a glimpse of the data
print("=" * 70)
print("PART A: DATA ACQUISITION AND UNDERSTANDING")
print("=" * 70)
print("\nFirst 5 observations:")
print(df.head())

# Report the number of observations (rows) and variables (columns) in the dataset
print(f"\nDataset shape: {df.shape}")
print(f"Number of observations: {df.shape[0]}")
print(f"Number of variables: {df.shape[1]}")

# Show the data types of each variable to understand their nature
print("\nData types:")
print(df.dtypes)

# Provide a detailed summary of the DataFrame, including non-null counts and memory usage
print("\nDataset info:")
df.info()

# Display a statistical summary of numerical columns (mean, std, min, max, quartiles)
print("\nStatistical summary:")
print(df.describe())

# Create a table describing each variable, its type, and a brief explanation
variable_desc = pd.DataFrame({
    'Variable': df.columns,
    'Type': df.dtypes.values,
    'Description': [
        'Age of beneficiary (years)',
        'Gender of beneficiary (male/female)',
        'Body Mass Index (kg/m²)',
        'Number of children/dependents',
        'Smoking status (yes/no)',
        'Residential region (US)',
        'Medical insurance charges ($) - TARGET VARIABLE'
    ]
})
print("\nVariable Description:")
print(variable_desc.to_string(index=False))

# --- PART B: DATA CLEANING AND PREPROCESSING ---

print("\n" + "=" * 70)
print("PART B: DATA CLEANING AND PREPROCESSING")
print("=" * 70)

# 1. Check for Missing Values
print("\n1. MISSING VALUES CHECK:")
missing_values = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Values': missing_values,
    'Percentage': missing_percentage
})
print(missing_df)

if missing_values.sum() == 0:
    print("\nNo missing values were found in the dataset.")
else:
    print(f"\nFound {missing_values.sum()} missing values.")

# 2. Check for Duplicate Records
print("\n2. DUPLICATE RECORDS CHECK:")
duplicates = df.duplicated().sum()
print(f"Number of duplicate records: {duplicates}")

if duplicates > 0:
    print("\nDuplicate records (first few):")
    print(df[df.duplicated(keep=False)].head())

    # Remove any identified duplicate records from the dataset
    df_clean = df.drop_duplicates().reset_index(drop=True)
    print(f"\nSuccessfully removed {duplicates} duplicate records.")
    print(f"New dataset shape: {df_clean.shape}")
else:
    df_clean = df.copy()
    print("No duplicate records were found.")

# 3. Outlier Detection
print("\n3. OUTLIER DETECTION:")

def detect_outliers_iqr(data, column):
    # Calculate the first quartile (Q1) and third quartile (Q3)
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    # Compute the Interquartile Range (IQR)
    IQR = Q3 - Q1
    # Determine the lower and upper bounds for outlier detection
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    # Identify outliers that fall outside these bounds
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

# Generate plots to visually inspect for outliers
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Create a boxplot for BMI distribution to show potential outliers
sns.boxplot(y=df_clean['bmi'], ax=axes[0], color='skyblue')
axes[0].set_title('BMI Distribution with Outliers')
axes[0].set_ylabel('BMI')

# Create a boxplot for Charges distribution to show potential outliers
sns.boxplot(y=df_clean['charges'], ax=axes[1], color='lightcoral')
axes[1].set_title('Charges Distribution with Outliers')
axes[1].set_ylabel('Charges ($)')

# Create a scatter plot of BMI vs Charges, colored by smoker status
sns.scatterplot(data=df_clean, x='bmi', y='charges', hue='smoker', alpha=0.6, ax=axes[2])
axes[2].set_title('BMI vs Charges')
axes[2].set_xlabel('BMI')
axes[2].set_ylabel('Charges ($)')

plt.tight_layout()
plt.show()

# Apply the IQR method to detect outliers in BMI and Charges columns
bmi_outliers, bmi_lower, bmi_upper = detect_outliers_iqr(df_clean, 'bmi')
charges_outliers, charges_lower, charges_upper = detect_outliers_iqr(df_clean, 'charges')

print(f"BMI outliers: {len(bmi_outliers)} ({len(bmi_outliers)/len(df_clean)*100:.2f}%)")
print(f"BMI normal range: [{bmi_lower:.2f}, {bmi_upper:.2f}]")
print(f"BMI extreme values: min={df_clean['bmi'].min():.2f}, max={df_clean['bmi'].max():.2f}")
print(f"\nCharges outliers: {len(charges_outliers)} ({len(charges_outliers)/len(df_clean)*100:.2f}%)")
print(f"Charges normal range: [{charges_lower:.2f}, {charges_upper:.2f}]")
print(f"Charges extreme values: min=${df_clean['charges'].min():.2f}, max=${df_clean['charges'].max():.2f}")

print("\nOutlier Treatment Decision:")
print("- BMI outliers: RETAINED - BMI can naturally be high in obese individuals, and these are legitimate data points.")
print("- Charges outliers: RETAINED - High charges are legitimate for smokers or individuals with significant health issues. Removing them would lead to a loss of valuable information about high-risk groups.")

# 4. Encoding Categorical Variables
print("\n4. ENCODING CATEGORICAL VARIABLES:")
print("\nUnique values for categorical variables before encoding:")
for col in ['sex', 'smoker', 'region']:
    print(f"{col}: {df_clean[col].unique()}")

# Apply One-Hot Encoding to convert categorical columns into numerical format
df_encoded = pd.get_dummies(df_clean, columns=['sex', 'smoker', 'region'], drop_first=True)

# Convert any new boolean columns created by one-hot encoding to integers (0 and 1)
bool_columns = df_encoded.select_dtypes(include=['bool']).columns
for col in bool_columns:
    df_encoded[col] = df_encoded[col].astype(int)

print(f"\nEncoding complete. The new dataset shape is: {df_encoded.shape}")
print("\nFirst 5 rows after encoding:")
print(df_encoded.head())

# 5. Feature Scaling Decision
print("\n5. FEATURE SCALING DECISION:")
print("\nVariable scales before any scaling:")
numeric_cols = ['age', 'bmi', 'children', 'charges']
for col in numeric_cols:
    print(f"{col}: min={df_encoded[col].min():.2f}, max={df_encoded[col].max():.2f}, mean={df_encoded[col].mean():.2f}")

print("\nFeature Scaling Decision:")
print("Linear Regression models do not strictly require feature scaling because:")
print("1. The coefficients in linear regression models automatically adjust to different scales of input features, ensuring interpretation remains valid.")
print("2. The Ordinary Least Squares (OLS) algorithm, commonly used in linear regression, directly minimizes the sum of squared errors and does not rely on gradient descent, which is where feature scaling is typically crucial.")
print("3. Retaining the original units of the features makes the regression coefficients directly interpretable in meaningful terms (e.g., dollar increase per year of age).")
print("\nDecision: No scaling applied - original units are preserved for better interpretability of coefficients.")

# --- PART C: EXPLORATORY DATA ANALYSIS (EDA) ---

print("\n" + "=" * 70)
print("PART C: EXPLORATORY DATA ANALYSIS")
print("=" * 70)

# 1. Summary Statistics
print("\n1. SUMMARY STATISTICS:")
print("\nDescriptive Statistics for Numeric Variables:")
print(df_clean.describe())

print("\nMean Charges by Smoker Status:")
print(df_clean.groupby('smoker')['charges'].describe())

print("\nMean Charges by Region:")
print(df_clean.groupby('region')['charges'].describe())

print("\nMean Charges by Sex:")
print(df_clean.groupby('sex')['charges'].describe())

# Calculate and display enhanced summary statistics including range and coefficient of variation
summary_stats = df_clean.describe().T
summary_stats['range'] = summary_stats['max'] - summary_stats['min']
summary_stats['cv'] = summary_stats['std'] / summary_stats['mean'] * 100
print("\nEnhanced Summary Statistics:")
print(summary_stats[['mean', 'std', 'min', 'max', 'range', 'cv']])

# 2. Distribution of Charges
print("\n2. DISTRIBUTION OF CHARGES:")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Create a histogram to visualize the frequency distribution of insurance charges
axes[0].hist(df_clean['charges'], bins=30, edgecolor='black', color='skyblue', alpha=0.7)
axes[0].set_title('Distribution of Insurance Charges', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Charges ($)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df_clean['charges'].mean(), color='red', linestyle='--', label=f'Mean: ${df_clean["charges"].mean():.0f}')
axes[0].axvline(df_clean['charges'].median(), color='green', linestyle='--', label=f'Median: ${df_clean["charges"].median():.0f}')
axes[0].legend()

# Create a boxplot to show the spread and identify potential outliers in charges
box_data = axes[1].boxplot(df_clean['charges'], vert=True, patch_artist=True)
box_data['boxes'][0].set_facecolor('lightblue')
axes[1].set_title('Boxplot of Charges', fontweight='bold')
axes[1].set_ylabel('Charges ($)')
axes[1].set_xticklabels(['Charges'])

# Create a density plot to visualize the probability density function of charges
sns.kdeplot(data=df_clean, x='charges', fill=True, ax=axes[2], alpha=0.6, color='skyblue')
axes[2].set_title('Density Plot of Charges', fontweight='bold')
axes[2].set_xlabel('Charges ($)')
axes[2].set_ylabel('Density')

plt.tight_layout()
plt.show()

# Calculate the skewness and kurtosis of the charges distribution
skewness = df_clean['charges'].skew()
kurtosis = df_clean['charges'].kurtosis()
print(f"Skewness: {skewness:.2f}")
print(f"Kurtosis: {kurtosis:.2f}")
print("\nInterpretation: The distribution of charges is right-skewed (positive skewness). This indicates that most individuals have lower medical charges, while a smaller group incurs very high charges. This pattern is expected, as factors like smoking status and age often lead to significantly higher medical costs.")

# 3. Relationship Analysis
print("\n3. RELATIONSHIP ANALYSIS:")

fig = plt.figure(figsize=(16, 12))

# Scatter plot showing the relationship between age and charges, distinguished by smoker status
ax1 = fig.add_subplot(2, 3, 1)
sns.scatterplot(data=df_clean, x='age', y='charges', hue='smoker', alpha=0.6, ax=ax1)
ax1.set_title('Age vs Charges (colored by smoker)', fontweight='bold')
ax1.set_xlabel('Age (years)')
ax1.set_ylabel('Charges ($)')
ax1.legend()

# Scatter plot showing the relationship between BMI and charges, distinguished by smoker status
ax2 = fig.add_subplot(2, 3, 2)
sns.scatterplot(data=df_clean, x='bmi', y='charges', hue='smoker', alpha=0.6, ax=ax2)
ax2.set_title('BMI vs Charges (colored by smoker)', fontweight='bold')
ax2.set_xlabel('BMI (kg/m²)')
ax2.set_ylabel('Charges ($)')
ax2.axvline(30, color='red', linestyle='--', alpha=0.5, label='Obese threshold (BMI=30)')
ax2.legend()

# Boxplot comparing charges between smokers and non-smokers
ax3 = fig.add_subplot(2, 3, 3)
sns.boxplot(data=df_clean, x='smoker', y='charges', ax=ax3, palette=['#2ecc71', '#e74c3c'])
ax3.set_title('Charges by Smoker Status', fontweight='bold')
ax3.set_xlabel('Smoker')
ax3.set_ylabel('Charges ($)')

# Boxplot comparing charges across different regions
ax4 = fig.add_subplot(2, 3, 4)
sns.boxplot(data=df_clean, x='region', y='charges', ax=ax4, palette='pastel')
ax4.set_title('Charges by Region', fontweight='bold')
ax4.set_xlabel('Region')
ax4.set_ylabel('Charges ($)')
ax4.tick_params(axis='x', rotation=45)

# Bar plot showing the average charges based on the number of children
ax5 = fig.add_subplot(2, 3, 5)
sns.barplot(data=df_clean, x='children', y='charges', ax=ax5, palette='viridis', errorbar=None)
ax5.set_title('Average Charges by Number of Children', fontweight='bold')
ax5.set_xlabel('Number of Children')
ax5.set_ylabel('Average Charges ($)')

# Heatmap displaying the correlation matrix for numerical variables
ax6 = fig.add_subplot(2, 3, 6)
numeric_df = df_clean[['age', 'bmi', 'children', 'charges']]
correlation_matrix = numeric_df.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, square=True, linewidths=1, ax=ax6, cbar_kws={"shrink": 0.8})
ax6.set_title('Correlation Matrix', fontweight='bold')

plt.tight_layout()
plt.show()

# Display detailed correlation values with charges, sorted in descending order
print("\nCorrelation with Charges:")
correlations = numeric_df.corr()['charges'].sort_values(ascending=False)
print(correlations)

# Perform an independent t-test to compare charges between smokers and non-smokers
smoker_charges = df_clean[df_clean['smoker'] == 'yes']['charges']
non_smoker_charges = df_clean[df_clean['smoker'] == 'no']['charges']
t_stat, p_value = stats.ttest_ind(smoker_charges, non_smoker_charges)
print(f"\nT-test for charges: Smokers vs Non-smokers")
print(f"T-statistic: {t_stat:.2f}")
print(f"P-value: {p_value:.10f}")
print(f"Interpretation: {'Smokers pay significantly more than non-smokers' if p_value < 0.05 else 'There is no significant difference in charges'}")

print(f"\nMean charges - Smokers: ${smoker_charges.mean():.2f}")
print(f"Mean charges - Non-smokers: ${non_smoker_charges.mean():.2f}")
print(f"Difference: ${smoker_charges.mean() - non_smoker_charges.mean():.2f}")

# --- PART D: LINEAR REGRESSION MODELING ---

print("\n" + "=" * 70)
print("PART D: LINEAR REGRESSION MODELING")
print("=" * 70)

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import statsmodels.api as sm

# Section 1: Simple Linear Regression
print("\n" + "=" * 60)
print("SECTION 1: SIMPLE LINEAR REGRESSION (BMI as predictor)")
print("=" * 60)

# Prepare the data by selecting 'bmi' as the independent variable and 'charges' as the dependent variable
X_simple = df_encoded[['bmi']]
y = df_encoded['charges']

# Fit a simple linear regression model using BMI to predict charges
simple_model = LinearRegression()
simple_model.fit(X_simple, y)

# Generate predictions for charges based on the simple linear model
y_pred_simple = simple_model.predict(X_simple)

# Extract the intercept and slope (coefficient) from the fitted model
intercept = simple_model.intercept_
slope = simple_model.coef_[0]
r2 = r2_score(y, y_pred_simple)

# Display the regression equation and the R-squared value
print(f"\nRegression Equation: charges = {intercept:.2f} + {slope:.2f} × bmi")
print(f"\nR² Value: {r2:.4f} ({r2*100:.2f}% of variance explained)")

# Use statsmodels for a more detailed statistical summary of the simple linear regression
X_simple_sm = sm.add_constant(X_simple)
simple_sm_model = sm.OLS(y, X_simple_sm).fit()
print("\nDetailed Statistics:")
print(simple_sm_model.summary().tables[1])

# Interpret the results of the simple linear regression model
print("\nINTERPRETATION:")
print(f"1. Intercept (${intercept:.2f}): This represents the predicted charges when BMI is zero. In a practical sense, a BMI of zero is not meaningful, so this value should be interpreted with caution and is largely a mathematical necessity for the model.")
print(f"2. Slope (${slope:.2f}): For every one-unit increase in BMI, the medical charges are predicted to increase by approximately $393.87, on average, holding all other factors constant.")
print(f"3. R² = {r2:.3f}: BMI alone explains only {r2*100:.1f}% of the variation in charges. This suggests that other factors, such as smoking status, are much more significant in determining medical charges.")

# Section 2: Multiple Linear Regression
print("\n" + "=" * 60)
print("SECTION 2: MULTIPLE LINEAR REGRESSION")
print("=" * 60)

# Prepare data for multiple regression by selecting all relevant feature columns
feature_cols = ['age', 'bmi', 'children', 'sex_male', 'smoker_yes', 'region_northwest', 'region_southeast', 'region_southwest']
X_multi = df_encoded[feature_cols]
y = df_encoded['charges']

# Add a constant term to the independent variables for the statsmodels OLS function
X_multi_sm = sm.add_constant(X_multi)

# Fit the multiple linear regression model using all selected features
multi_model = sm.OLS(y, X_multi_sm).fit()

# Display the comprehensive results of the multiple linear regression model
print("\n" + "=" * 70)
print("MULTIPLE LINEAR REGRESSION RESULTS")
print("=" * 70)
print(multi_model.summary())

# Create a DataFrame to summarize the coefficients, standard errors, t-values, and p-values
coef_table = pd.DataFrame({
    'Variable': ['Constant'] + feature_cols,
    'Coefficient': multi_model.params.values,
    'Std. Error': multi_model.bse.values,
    't-value': multi_model.tvalues.values,
    'p-value': multi_model.pvalues.values
})
# Indicate whether each coefficient is statistically significant at the 0.05 level
coef_table['Significant'] = coef_table['p-value'] < 0.05

print("\n" + "=" * 70)
print("COEFFICIENT TABLE")
print("=" * 70)
print(coef_table.to_string(index=False, float_format=lambda x: '%.4f' % x))

# Construct and display the full regression equation based on the model's coefficients
print("\n" + "=" * 70)
print("REGRESSION EQUATION")
print("=" * 70)
equation = "charges = "
for i, var in enumerate(['const'] + feature_cols):
    coef = multi_model.params[i]
    if i == 0:
        equation += f"{coef:.2f}"
    else:
        sign = "+" if coef >= 0 else "-"
        equation += f" {sign} {abs(coef):.2f}×{var}"
print(equation)

# Provide key interpretations of the multiple linear regression model's findings
print("\n" + "=" * 70)
print("KEY INTERPRETATIONS")
print("=" * 70)

print("\nSTATISTICALLY SIGNIFICANT VARIABLES (p < 0.05):")
significant_vars = coef_table[coef_table['Significant']]['Variable'].tolist()
for var in significant_vars:
    if var != 'Constant':
        print(f"- {var}")

# Calculate standardized coefficients to allow for fair comparison of variable importance
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_encoded[feature_cols])
y_scaled = (y - y.mean()) / y.std()
scaled_model = LinearRegression().fit(X_scaled, y_scaled)
std_coef = pd.DataFrame({
    'Variable': feature_cols,
    'Standardized Coef': scaled_model.coef_
}).sort_values('Standardized Coef', ascending=False)

print("\nSTANDARDIZED COEFFICIENTS (for fair comparison of impact):")
print(std_coef.to_string(index=False))

print("\nIMPACT OF SMOKING ON CHARGES:")
smoker_coef = multi_model.params['smoker_yes']
print(f"Holding all other factors constant, individuals who smoke are predicted to pay ${smoker_coef:,.2f} more than non-smokers (the baseline group).")
print(f"This represents a {smoker_coef/y.mean()*100:.1f}% increase over the average charge.")

print(f"\nModel Fit:")
print(f"R²: {multi_model.rsquared:.4f} ({multi_model.rsquared*100:.1f}% of variance explained)")
print(f"Adjusted R²: {multi_model.rsquared_adj:.4f}")

# --- PART E: MODEL EVALUATION AND ASSUMPTIONS ---

print("\n" + "=" * 70)
print("PART E: MODEL EVALUATION AND ASSUMPTIONS")
print("=" * 70)

from sklearn.model_selection import train_test_split

# 1. Train-Test Split
print("\n1. TRAIN-TEST SPLIT:")
# Split the data into training and testing sets to evaluate model performance on unseen data
X_train, X_test, y_train, y_test = train_test_split(
    X_multi, y, test_size=0.3, random_state=42
)

print(f"Training set size: {X_train.shape[0]} observations ({len(X_train)/len(X_multi)*100:.1f}% of total data)")
print(f"Test set size: {X_test.shape[0]} observations ({len(X_test)/len(X_multi)*100:.1f}% of total data)")

# Fit the linear regression model using only the training data
train_model = LinearRegression()
train_model.fit(X_train, y_train)

# Generate predictions for both the training and testing datasets
y_train_pred = train_model.predict(X_train)
y_test_pred = train_model.predict(X_test)

# 2. Model Performance Metrics
print("\n2. MODEL PERFORMANCE METRICS:")

# Calculate key performance metrics: Root Mean Squared Error (RMSE), Mean Absolute Error (MAE), and R-squared (R²)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

# Create a table to display the calculated performance metrics for both training and test sets
metrics_df = pd.DataFrame({
    'Metric': ['RMSE ($)', 'MAE ($)', 'R² Score'],
    'Training Set': [f"${train_rmse:,.2f}", f"${train_mae:,.2f}", f"{train_r2:.4f}"],
    'Test Set': [f"${test_rmse:,.2f}", f"${test_mae:,.2f}", f"{test_r2:.4f}"],
    'Difference': [
        f"${abs(test_rmse - train_rmse):,.2f}",
        f"${abs(test_mae - train_mae):,.2f}",
        f"{abs(test_r2 - train_r2):.4f}"
    ]
})

print("\n" + "=" * 70)
print(metrics_df.to_string(index=False))
print("=" * 70)

print("\nPERFORMANCE INTERPRETATION:")
print(f"- The model successfully explains {test_r2*100:.1f}% of the variance in medical charges within the test data.")
print(f"- The average prediction error (MAE) is approximately ${test_mae:,.2f}, indicating the typical magnitude of prediction inaccuracies.")
print(f"- The RMSE (${test_rmse:,.2f}) being greater than the MAE (${test_mae:,.2f}) suggests that there are some larger prediction errors, as RMSE penalizes larger errors more heavily.")
print(f"- A small difference between training and testing set performance metrics indicates that the model generalizes well to new, unseen data, suggesting it is not severely overfitting.")

# 3. Regression Assumptions Check
print("\n3. REGRESSION ASSUMPTIONS CHECK:")

# Calculate residuals (actual minus predicted values) and standardized residuals
residuals = y_test - y_test_pred
standardized_residuals = (residuals - residuals.mean()) / residuals.std()

# Create a set of diagnostic plots to visually inspect regression assumptions
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Plot 1: Linearity Check - Actual vs. Predicted values
axes[0, 0].scatter(y_test_pred, y_test, alpha=0.6, color='steelblue')
axes[0, 0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2, label='Perfect Prediction')
axes[0, 0].set_xlabel('Predicted Charges ($)')
axes[0, 0].set_ylabel('Actual Charges ($)')
axes[0, 0].set_title('Linearity Check: Actual vs Predicted', fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Homoscedasticity Check - Residuals vs. Fitted values
axes[0, 1].scatter(y_test_pred, residuals, alpha=0.6, color='steelblue')
axes[0, 1].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0, 1].set_xlabel('Fitted Values ($)')
axes[0, 1].set_ylabel('Residuals ($)')
axes[0, 1].set_title('Homoscedasticity Check: Residuals vs Fitted', fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Add a trend line to the residuals vs fitted plot to highlight any pattern
z = np.polyfit(y_test_pred, residuals, 1)
p = np.poly1d(z)
axes[0, 1].plot(sorted(y_test_pred), p(sorted(y_test_pred)), "g-", linewidth=2, label='Trend line')
axes[0, 1].legend()

# Plot 3: Normality Check - Histogram of Residuals
axes[0, 2].hist(residuals, bins=30, edgecolor='black', color='steelblue', alpha=0.7, density=True)
axes[0, 2].set_xlabel('Residuals ($)')
axes[0, 2].set_ylabel('Density')
axes[0, 2].set_title('Normality Check: Residual Distribution', fontweight='bold')

# Overlay a normal distribution curve on the histogram for comparison
x = np.linspace(residuals.min(), residuals.max(), 100)
axes[0, 2].plot(x, stats.norm.pdf(x, residuals.mean(), residuals.std()), 'r-', linewidth=2, label='Normal Distribution')
axes[0, 2].legend()

# Plot 4: Q-Q Plot for Normality assessment of residuals
stats.probplot(residuals, dist="norm", plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot: Normality Check', fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# Plot 5: Scale-Location Plot (Square root of standardized residuals vs. Fitted values)
axes[1, 1].scatter(y_test_pred, np.sqrt(np.abs(standardized_residuals)), alpha=0.6, color='steelblue')
axes[1, 1].set_xlabel('Fitted Values ($)')
axes[1, 1].set_ylabel('√|Standardized Residuals|')
axes[1, 1].set_title('Scale-Location Plot', fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

# Plot 6: Residuals vs. Order of observations to check for independence
axes[1, 2].scatter(range(len(residuals)), residuals, alpha=0.6, color='steelblue')
axes[1, 2].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[1, 2].set_xlabel('Observation Order')
axes[1, 2].set_ylabel('Residuals ($)')
axes[1, 2].set_title('Residuals vs Order', fontweight='bold')
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 4. Multicollinearity Check (VIF)
print("\n4. MULTICOLLINEARITY CHECK (VIF):")

from statsmodels.stats.outliers_influence import variance_inflation_factor

# Calculate Variance Inflation Factor (VIF) for each predictor variable
vif_data = pd.DataFrame()
vif_data['Variable'] = feature_cols
vif_data['VIF'] = [variance_inflation_factor(X_multi.values, i) for i in range(X_multi.shape[1])]

vif_data['Interpretation'] = vif_data['VIF'].apply(
    lambda x: 'Severe multicollinearity' if x > 10
    else ('Moderate multicollinearity' if x > 5
          else 'No serious multicollinearity')
)

print("\n" + "=" * 70)
print(vif_data.to_string(index=False))
print("=" * 70)

print("\nVIF Interpretation Guidelines:")
print("- VIF < 5: Generally indicates no serious multicollinearity among predictors.")
print("- VIF 5-10: May indicate moderate multicollinearity, which could warrant further investigation.")
print("- VIF > 10: Suggests severe multicollinearity, potentially requiring removal or transformation of correlated variables.")

# 5. Assumptions Summary
print("\n5. ASSUMPTIONS SUMMARY:")

print("\n" + "=" * 70)
print("REGRESSION ASSUMPTIONS SUMMARY")
print("=" * 70)

# 1. Linearity Assessment
print("\n1. LINEARITY:")
if test_r2 > 0.7:
    print("   - The model shows a good linear relationship with strong predictive power.")
else:
    print("   - The model exhibits a moderate linear relationship, suggesting some non-linearity might be present.")

# 2. Homoscedasticity Assessment
residual_var_by_fitted = pd.Series(residuals).groupby(pd.cut(y_test_pred, 10)).var()
if not residual_var_by_fitted.isna().all() and residual_var_by_fitted.std() / residual_var_by_fitted.mean() < 0.5:
    print("   - The assumption of homoscedasticity appears to be satisfied, meaning residual variance is relatively constant.")
else:
    print("   - Some heteroscedasticity is present, indicating that the variance of residuals is not constant across all fitted values.")

# 3. Normality of Residuals Assessment (using Shapiro-Wilk test if sample size allows)
if len(residuals) > 5000:
    shapiro_sample = residuals.sample(5000)
else:
    shapiro_sample = residuals
shapiro_stat, shapiro_p = stats.shapiro(shapiro_sample)
if shapiro_p > 0.05:
    print("   - The residuals are approximately normally distributed (Shapiro-Wilk p-value > 0.05).")
else:
    print("   - The residuals deviate from normality. However, linear regression is often robust to moderate violations of this assumption, especially with larger sample sizes.")

# 4. Multicollinearity Assessment
if vif_data['VIF'].max() < 5:
    print("   - No serious multicollinearity issues were detected among the predictor variables.")
elif vif_data['VIF'].max() < 10:
    print("   - Moderate multicollinearity was detected in some variables, which might be worth noting.")
else:
    print("   - Severe multicollinearity is present. It is recommended to consider removing or transforming highly correlated variables.")

print("\nOVERALL ASSESSMENT:")
print("The linear regression model generally satisfies the key assumptions required for valid statistical inference. While some heteroscedasticity and deviations from normality are observed in the residuals, these are not severe enough to invalidate the model, especially considering the dataset's characteristics and practical context.")

# --- PART F: INTERPRETATION AND BUSINESS RECOMMENDATIONS ---

print("\n" + "=" * 70)
print("PART F: INTERPRETATION AND BUSINESS RECOMMENDATIONS")
print("=" * 70)

# 1. Factor Analysis
print("\n1. FACTOR ANALYSIS - IMPACT ON MEDICAL CHARGES:")

coef_df = pd.DataFrame({
    'Variable': feature_cols,
    'Coefficient': multi_model.params[1:].values,
    'P-Value': multi_model.pvalues[1:].values
})

# Add standardized coefficients to compare the relative impact of each variable
coef_df['Std_Coefficient'] = std_coef['Standardized Coef'].values
coef_df['Abs_Std_Coef'] = abs(coef_df['Std_Coefficient'])
coef_df = coef_df.sort_values('Abs_Std_Coef', ascending=False)

print("\nWHICH FACTOR INCREASES CHARGES THE MOST?")
top_factor = coef_df.iloc[0]
print(f"\n- The variable **{top_factor['Variable']}** exhibits the strongest influence on medical charges.")
print(f"  Standardized coefficient: {top_factor['Std_Coefficient']:.3f}")
print(f"  This implies that a one-standard deviation increase in {top_factor['Variable']} leads to an increase in charges by {top_factor['Std_Coefficient']:.3f} standard deviations, assuming other factors are held constant.")

print("\nFACTOR RANKING (by standardized impact on charges):")
for i, row in coef_df.iterrows():
    impact = "positive" if row['Std_Coefficient'] > 0 else "-"
    sig = "(Statistically Significant)" if row['P-Value'] < 0.05 else "(Not Statistically Significant)"
    print(f"  {i+1}. {row['Variable']}: {row['Std_Coefficient']:.3f} ({impact} impact) {sig}")

# 2. BMI Impact Analysis
print("\n2. BMI IMPACT ANALYSIS:")

bmi_coef = multi_model.params['bmi']
bmi_pvalue = multi_model.pvalues['bmi']

print(f"\nIf BMI increases by 1 unit, the expected change in charges is:")
print(f"\n   Expected change = ${bmi_coef:.2f}")
print(f"   (p-value: {bmi_pvalue:.4f})")

print(f"\nPractical examples of BMI's influence on charges:")
print(f"   - For an individual whose BMI increases from 25 to 26: charges are expected to increase by ${bmi_coef:.2f}.")
print(f"   - For an individual whose BMI increases from 30 to 35: charges are expected to increase by ${bmi_coef*5:.2f}.")
print(f"   - For an individual moving from a normal BMI (e.g., 22) to an obese BMI (e.g., 32): the estimated increase in charges would be ${bmi_coef*10:.2f}.")

if bmi_pvalue < 0.05:
    print(f"\nThis effect is statistically significant, suggesting a reliable relationship between BMI and charges.")
else:
    print(f"\nThis effect is not statistically significant, meaning the observed relationship could be due to random chance.")

# 3. Business Recommendations
print("\n3. BUSINESS RECOMMENDATIONS:")
print("\n" + "=" * 70)

recommendations = [
    {
        'title': '1. Implement Smoking Cessation Programs',
        'finding': f"Smokers incur approximately ${multi_model.params['smoker_yes']:,.2f} more in medical charges on average compared to non-smokers.",
        'recommendation': "Offer attractive premium discounts for non-smokers and actively promote and subsidize smoking cessation programs. This strategy benefits both the insurer (by reducing future claims) and the insured (by improving health and lowering costs).",
        'impact': "High - Potential to significantly reduce claims, possibly by 30-40% for participating members who successfully quit smoking."
    },
    {
        'title': '2. Introduce BMI-Based Premium Tiers and Wellness Incentives',
        'finding': f"Each one-unit increase in BMI is associated with an increase of ${bmi_coef:.2f} in medical charges.",
        'recommendation': "Develop premium structures that reflect BMI categories (e.g., normal, overweight, obese). Complement this with wellness programs that offer incentives for maintaining a healthy BMI, such as discounts on gym memberships or health coaching services.",
        'impact': "Medium - Provides incremental improvement in risk assessment and encourages healthier lifestyles, leading to more accurate pricing and potentially reduced health risks."
    },
    {
        'title': '3. Tailor Product Offerings and Marketing by Age Group',
        'finding': f"Medical charges increase by approximately ${multi_model.params['age']:.2f} for each additional year of age.",
        'recommendation': "Design age-specific insurance products and marketing campaigns. For younger clients, focus on preventive care and healthy lifestyle packages. For older clients, emphasize comprehensive coverage for chronic condition management and specialized care.",
        'impact': "Medium - Enhances customer segmentation, allowing for more relevant product offerings and potentially higher customer satisfaction and retention."
    }
]

for rec in recommendations:
    print(f"\n{rec['title']}")
    print(f"   - Key Finding: {rec['finding']}")
    print(f"   - Recommendation: {rec['recommendation']}")
    print(f"   - Expected Impact: {rec['impact']}")

print("\nADDITIONAL RECOMMENDATIONS:")
print("   4. Utilize the developed model to ensure fair and equitable premium calculations across different geographical regions.")
print("   5. Consider developing family-oriented insurance plans that account for the number of children, as this is a factor influencing charges.")
print("   6. Proactively monitor and engage with high-risk groups, such as smokers and individuals with high BMI, through dedicated health management programs.")

# --- SUMMARY ---

print("\n" + "=" * 70)
print("ANALYSIS COMPLETE - SUMMARY")
print("=" * 70)
print(f"\nDataset Analysis: The cleaned dataset comprises {len(df_clean)} observations and {len(df_clean.columns)} variables.")
print(f"Model Performance: The linear regression model achieved an R² score of {test_r2:.4f} on the test data, indicating its predictive power.")
print(f"Key Predictor: Smoking status was identified as the most significant predictor, with smokers incurring approximately ${multi_model.params['smoker_yes']:,.2f} more in charges.")
print(f"Average Prediction Error: The model's average prediction error (MAE) on the test set is ${test_mae:,.2f}.")
print("\n" + "=" * 70)